# Module 2: Deploy — Managed Deep Agents

> Part of the **Modular Workshops** series. Standalone, ~15 min.

Module 1 built an agent by hand, piece by piece, with `create_deep_agent()` — running inside this notebook's own Python process. This module takes that same agent and ships it as a **Managed Deep Agent (MDA)**: a hosted, production-ready deployment with a REST API, a Studio UI, durable memory, and more — no hand-built backend, store, checkpointer, or server required.

We'll walk through the deployable project at `agents/deep_agent/`, map it back to the `create_deep_agent()` calls from Module 1, run it locally with `mda dev`, then ship it with `mda deploy`.

## Setup

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:   ", bool(os.environ.get("OPENAI_API_KEY")))

LANGSMITH_API_KEY set: True
OPENAI_API_KEY set:    True


---
# From Notebook to Managed Deployment

`define_deep_agent` replaces `create_deep_agent`: instead of building the agent object yourself, you hand the managed runtime a *definition*, and it builds, hosts, and serves the actual agent on LangSmith's Agent Server — no backend, store, checkpointer, or server process to run yourself.

## 1. Project structure

A Managed Deep Agents project is a directory with an `agent.py` at its root — `agents/deep_agent/`, in this case. Unlike the notebook-built agents in Module 1, it's a self-contained project with its own `pyproject.toml` and `.env`, so its dependencies and the `mda` CLI stay isolated from this workshop's shared environment.

In [2]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("agents/deep_agent/")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ and build output for clarity
    dirs[:] = [d for d in dirs if d not in ("__pycache__", ".mda")]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

agents/deep_agent/
---
deep_agent/
  uv.lock
  memory.py
  pyproject.toml
  README.md
  agent.py
  .env.example
  instructions.md
  tools/
    search.py
  .venv/
    .lock
    pyvenv.cfg
    CACHEDIR.TAG
    .gitignore
    bin/
      jsondiff
      activate.bat
      activate.ps1
      python3
      python
      distro
      activate.fish
      idna
      websockets
      pydoc.bat
      filetype
      activate_this.py
      mda
      httpx
      cffi-gen-src
      activate
      activate.nu
      normalizer
      jsonpointer
      jsonpatch
      deactivate.bat
      httpx2
      python3.14
      activate.csh
    lib/
      python3.14/
        site-packages/
          jsonpatch.py
          jsonpointer.py
          _virtualenv.py
          _virtualenv.pth
          jsonpatch_cli.py
          _cffi_backend.cpython-314-darwin.so
          typing_extensions.py
          orjson/
            __init__.pyi
            __init__.py
            orjson.cpython-314-darwin.so
            py.typed


## 2. From `create_deep_agent` to `define_deep_agent`

The agent you built across Module 1 maps onto this project as follows:

| Module 1 (`create_deep_agent`) | `agents/deep_agent/` (MDA) |
|---|---|
| `system_prompt=...`, `memory=["/AGENTS.md"]` | `instructions.md` |
| `skills=["/skills/"]` | `skills/<name>/SKILL.md` (same files) |
| Custom `StoreBackend` route for `/memories/` | `memory.py` -> `define_memory(agent=MemoryLayer())`, mounted at `/memories/agent/` |
| `FilesystemBackend(virtual_mode=True)` for `/final_report.md` | No backend to declare — the managed runtime gives every run the same thread-scoped virtual filesystem by default |
| `checkpointer=MemorySaver()` | No equivalent to declare — threads and persistence come from Agent Server |
| `model`, `tools`, `subagents`, `interrupt_on` | Carry over unchanged |

`name` is new and required — a static string that becomes the agent's graph ID and the default deployment name.

**Docs:** [Managed Deep Agents overview](https://docs.langchain.com/langsmith/python/managed-deep-agents-overview) · [Move from Deep Agents](https://docs.langchain.com/langsmith/python/managed-deep-agents-migrate) · [Agent definition](https://docs.langchain.com/langsmith/python/managed-deep-agents-agent-definition)

## 3. agent.py + instructions.md

`agent.py` must export a module-level `agent` variable — that's what `mda build`, `mda dev`, and `mda deploy` all load. `instructions.md` holds the system prompt; it syncs to [Context Hub](https://docs.langchain.com/langsmith/python/managed-deep-agents-context-hub) on deploy, where you can edit it later without redeploying.

In [3]:
agent_path = os.path.join(agent_dir, "agent.py")
with open(agent_path) as f:
    print("agent.py:")
    print(f.read())

instructions_path = os.path.join(agent_dir, "instructions.md")
with open(instructions_path) as f:
    print("\ninstructions.md:")
    print(f.read())

agent.py:
"""The Module 2 deployable research agent, built with Managed Deep Agents (MDA).

Module 1 builds this agent by hand, piece by piece, with `create_deep_agent`.
This project packages the same harness as a Managed Deep Agent: `mda deploy`
builds, hosts, and serves it on LangSmith's Agent Server -- no hand-built
backend, store, checkpointer, or server to run yourself.

Mapping from `create_deep_agent` to `define_deep_agent`, per the migration
guide (https://docs.langchain.com/langsmith/python/managed-deep-agents-migrate):

  - `system_prompt` + the `AGENTS.md` memory file -> `instructions.md`
    (synced to Context Hub on `mda deploy`)
  - `skills=["./skills/"]` -> `skills/<name>/SKILL.md` (same files, synced to
    Context Hub)
  - a hand-built `/memories/` `StoreBackend` route -> `memory.py`'s
    `define_memory(agent=MemoryLayer())`, mounted at `/memories/agent/`
  - a hand-built `backend=CompositeBackend(FilesystemBackend(virtual_mode=True), ...)`
    -> no backend to declar

`memory.py` declares the durable memory layer mounted at `/memories/agent/` — the MDA equivalent of the `StoreBackend` route you'd otherwise wire up by hand. See §2 above for the full parameter mapping.

## 4. Local development

Three CLI commands you'll use (all run from `agents/deep_agent/`, where its own `pyproject.toml` and `agent.py` live):

```bash
# Compile the project into a managed LangGraph app, without deploying
uv run mda build

# Compile and run on the local LangGraph dev server (Studio UI + hot reload)
uv run mda dev

# Compile, sync Context Hub, upload, and deploy to LangSmith
uv run mda deploy
```

`mda dev` opens LangSmith Studio in your browser — useful to step through tool calls and approve HITL interrupts visually, same as `langgraph dev` would for a hand-built graph.

**Docs:** [Managed Deep Agents CLI reference](https://docs.langchain.com/langsmith/python/managed-deep-agents-cli) · [Local development](https://docs.langchain.com/langsmith/python/managed-deep-agents-local-development).

## 5. Build the project

`mda build` compiles the project into a managed LangGraph app (into `.mda/build`) — no Context Hub sync, no upload, no deploy. Use it to catch import and config errors before deploying.

In [4]:
# `cd` and `!` run in a subshell -- chain in one line so cwd applies to the mda command.
!cd "{project_root}/agents/deep_agent" && uv sync && uv run mda build

Resolved 62 packages in 2ms
Checked 60 packages in 12ms

◇  Managed app compiled
│  /Users/emilypoland/langchain-garage-week/agents/deep_agent/.mda/build
│  
◇  Build warnings
│  memory.py enables agent memory, but .mda/__contexthub__/memories/agent/ is empty. Add initial memory there if needed.
│  
◇  Run locally
│  mda dev
│  
◇  Deploy when ready
└  mda deploy



## 6. Deploy to LangSmith (optional)

Run the cell below to deploy with MDA. `mda deploy` compiles the project, syncs `instructions.md` and `skills/` to Context Hub, uploads the project, and deploys it to the same [Agent Server](https://docs.langchain.com/langsmith/agent-server-overview) runtime LangSmith Deployments use. Provisioning takes a few minutes.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. `mda deploy` creates a **Serverless** deployment by default, which requires a supported GCP SaaS region; use `--deployment-type dedicated` for always-on capacity elsewhere (AWS, Azure, self-hosted, hybrid).

> Everything in `agents/deep_agent/` becomes part of the build archive. `mda build` forwards non-reserved `.env` values to the deployment as secrets and excludes `.env` itself from the archive — being in `.gitignore` does not exclude it.

> **Outside the US?** `mda deploy` targets the GCP US SaaS region by default (`LANGSMITH_ENDPOINT=https://api.smith.langchain.com`). If your LangSmith org/workspace is provisioned in a different SaaS region, your API key will not authenticate against the US endpoint — set `LANGSMITH_ENDPOINT` in `agents/deep_agent/.env` to match your org before deploying: `https://eu.api.smith.langchain.com` (GCP EU), `https://apac.api.smith.langchain.com` (GCP APAC), or `https://aws.api.smith.langchain.com` (AWS US). This is about which region your workspace lives in, not where you physically run the notebook from.

Useful flags:
- `--name <name>` — deployment name (defaults to the `name` passed to `define_deep_agent`, here `deep-agent`)
- `--deployment-type dedicated` — always-on deployment; required outside GCP SaaS regions (default is `serverless`)
- `--no-wait` — trigger the build without polling for completion

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type dedicated` for always-on capacity outside GCP SaaS regions.
!cd "{project_root}/agents/deep_agent" && uv run mda deploy


 ⡷⢾ ⣎⣱ ⡷⣸ ⣎⣱ ⡎⠑ ⣏⡉ ⡏⢱           
 ⠇⠸ ⠇⠸ ⠇⠹ ⠇⠸ ⠣⠝ ⠧⠤ ⠧⠜           
 ⡏⢱ ⣏⡉ ⣏⡉ ⣏⡱   ⣎⣱ ⡎⠑ ⣏⡉ ⡷⣸ ⢹⠁ ⢎⡑
 ⠧⠜ ⠧⠤ ⠧⠤ ⠇    ⠇⠸ ⠣⠝ ⠧⠤ ⠇⠹ ⠸  ⠢⠜
┌  v0.10.1
│
●  Project: /Users/emilypoland/langchain-garage-week/agents/deep_agent
│  
●  Organization: Deployed Engineering
│  
◇  No existing deployment — creating a new one                                  │                                                                               ◓  Compiling project                                                            
│  
●  Workspace: Emily Poland's Workspace00-97d0-166cc757ae49                      │                                                                               ◓  Creating deployment 'deep-agent'                                             
│  
◇  Deployment dashboard
https://smith.langchain.com/o/22e7329b-39d3-45f3-baf7-2ab127e18246/host/deployments/6e22fb30-c5a1-4b00-97d0-166cc757ae49
│
◐  DEPLOYING…                                                                   │                   

## 7. What you get with Managed Deep Agents

Once deployed, your agent runs on the same [Agent Server](https://docs.langchain.com/langsmith/agent-server-overview) as a hand-built LangSmith Deployment, reachable through 30+ endpoints, plus the managed layer MDA adds on top:

| Capability | What you get |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Context Hub** | Edit `instructions.md` and skills live, without a redeploy |
| **Durable memory** | `memory.py` -> `/memories/agent/`, survives restarts and threads |
| **HITL** | Interrupt and resume from any client |
| **Channels** | Wire the agent up to Slack and other messaging surfaces declaratively |
| **Schedules** | Managed cron — trigger your agent on a recurring basis |

## Recap

| Step | What | How |
|---|---|---|
| **Map** | `create_deep_agent` → `define_deep_agent` | `instructions.md`, `skills/`, `memory.py` replace hand-built prompt, skills, and backend wiring |
| **Structure** | Project layout | `agent.py`, `instructions.md`, `skills/`, `memory.py` in `agents/deep_agent/` |
| **Define** | Declare the agent | `define_deep_agent(name=..., model=..., tools=..., subagents=..., interrupt_on=...)` |
| **Run locally** | Studio + hot reload | `mda dev` |
| **Ship** | Deploy to a managed Agent Server | `mda deploy` |

**The production pattern:** the same agent you built in Module 1, declared instead of constructed, shipped with one command. It now lives behind a managed server with a REST API, durable memory, and 30+ endpoints available to call it.

**Next:** Module 3 — LangSmith (tracing, querying traces, offline + online evals, annotation queues).

## Clean up (optional)

Run the cell below to tear down the deployment and the LangSmith resources `mda deploy` created for it -- `mda destroy` is an alias for the same command. This only removes the hosted deployment; your local project files in `agents/deep_agent/` are untouched.

> `--yes` skips the interactive delete confirmation, which a notebook `!` cell can't answer. Drop it to confirm from a terminal instead.

**Docs:** [Delete deployments](https://docs.langchain.com/langsmith/python/managed-deep-agents-cli#delete-deployments).

In [12]:
# Deletes the deployment named "deep-agent" (the `name` passed to define_deep_agent) in this workspace.
# Pass --name <name> here too if you deployed with a different --name in §6 above.
!cd "{project_root}/agents/deep_agent" && uv run mda delete --yes


 ⡷⢾ ⣎⣱ ⡷⣸ ⣎⣱ ⡎⠑ ⣏⡉ ⡏⢱           
 ⠇⠸ ⠇⠸ ⠇⠹ ⠇⠸ ⠣⠝ ⠧⠤ ⠧⠜           
 ⡏⢱ ⣏⡉ ⣏⡉ ⣏⡱   ⣎⣱ ⡎⠑ ⣏⡉ ⡷⣸ ⢹⠁ ⢎⡑
 ⠧⠜ ⠧⠤ ⠧⠤ ⠇    ⠇⠸ ⠣⠝ ⠧⠤ ⠇⠹ ⠸  ⠢⠜
┌  v0.10.1
│
●  Project: /Users/emilypoland/langchain-garage-week/agents/deep_agent
│  
▲  Will delete the LangSmith deployment and its tracing project                 │                                                                               ◐  Looking up sandbox registry                                                  
│  
▲  Will delete Context Hub repo 'deep-agent'
│  
└  Deleted 'deep-agent'.o(s)                                                │                                                                               ◒  Deleting Context Hub repo 'deep-agent'                                       
